In [16]:
# Sam Brown
# Sam_brown@mines.edu
# June 20
# Goal: Use LSTM neural nets to capture and leverage long and short term patterns in the tidal modulation

import sys
sys.path.append("/Users/sambrown04/Documents/SURF/whillans-surf/notebooks/SURF")

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split, TensorDataset
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

df = pd.read_csv("/Users/sambrown04/Documents/SURF/Preproc_data/10-18.csv", parse_dates=["start_time"])
df = df.iloc[500:3000] # NANS UNKNOWN CHECK AGAIN

In [17]:
df.head()

,tide_deriv,form_fac,time_since,slip_size_standardized,high_t_evt,start_time,sev_stds,pre-s_stds,tide_height,inter_form_Fac,A_diurn,A_semidiurn
500,-0.314791,1.280039,613.75,-1.013850,0,2010-12-13 19:05:00,-0.909179,-0.176430,-6.550971,0.967561,23.603771,24.395125
501,0.141072,1.237633,964.25,-0.147460,1,2010-12-14 11:10:00,-0.348784,0.174002,7.011898,1.285889,-27.136151,-21.103021
502,0.063172,1.692694,1475.00,1.558077,1,2010-12-15 11:45:00,0.880731,0.702934,16.689716,1.487838,35.066599,23.568823
503,-0.025756,2.579601,1440.00,1.681079,1,2010-12-16 11:45:00,1.182876,0.401941,29.228166,2.099411,46.453259,22.126808
504,-0.074856,4.106137,1468.00,1.867888,1,2010-12-17 12:13:00,1.425515,0.955393,45.560761,3.113317,59.218770,19.021120


In [45]:
# Features and Target
X = df[['tide_height', 'tide_deriv', 'A_diurn', 'A_semidiurn', 'high_t_evt', 'time_since', 'sev_stds', 'pre-s_stds', 'form_fac']]
y = df['slip_size_standardized'].values.reshape(-1,1)

# Normalize features
x_scaler = StandardScaler()
X_scaled = x_scaler.fit_transform(X)

y_scaler = StandardScaler()
y_scaled = y_scaler.fit_transform(y)

In [46]:
SEQ_LEN =  30 # Sequence of "memory"
batch_size = 32

sequences = []
targets = []

# We Want to create rolling sequences of SEQ_LEN
for i in range(len(X_scaled) - SEQ_LEN):
    seq = X_scaled[i:i+SEQ_LEN]
    target = y_scaled[i + SEQ_LEN - 1][0]  # flatten from 2D array NOTE: To change which target we predict remove -1 -> predicts target after last row of features.
    sequences.append(seq)
    targets.append(target)

#Tensors
X_seq = torch.tensor(np.array(sequences), dtype=torch.float32)
y_seq = torch.tensor(np.array(targets), dtype=torch.float32).unsqueeze(1)

print("X_seq shape:", X_seq.shape)  # [num_samples, seq_len, 6]
print("y_seq shape:", y_seq.shape)  # [num_samples, 1]

X_seq shape: torch.Size([2470, 30, 9])
y_seq shape: torch.Size([2470, 1])


In [14]:
# # Wrap to Tensor Dataset
# dataset = TensorDataset(X_seq, y_seq)

# # train test 80 20
# train_size = int(0.8 * len(dataset))
# test_size = len(dataset) - train_size

# train_set, test_set = random_split(dataset, [train_size, test_size])

# # Create DataLoaders
# train_loader = DataLoader(train_set, batch_size, shuffle=True) # splits into mini-batches
# test_loader = DataLoader(test_set, batch_size, shuffle=False)

train_size = int(0.8 * len(X_seq))

X_train = X_seq[:train_size]
y_train = y_seq[:train_size]
X_test = X_seq[train_size:]
y_test = y_seq[train_size:]

# Create DataLoaders with no shuffling
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=32, shuffle=False)

NameError: name 'X_seq' is not defined

In [68]:
# Model
class SlipLSTM(nn.Module):
    def __init__(self, input_size=9, hidden_size=64, num_layers=1):
        super(SlipLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            batch_first=True)

        self.fc1 = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.3) # Dropout, saw on forum... prevents overfitiing
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.lstm(x)       # out: [batch, seq_len, hidden_size]
        out = out[:, -1, :]         # take output at last time step
        out = self.fc1(out)         # linear layer 1
        out = self.relu(out)        # ReLU activation
        out = self.fc2(out)         # final output layer
        return out

In [69]:
model = SlipLSTM(input_size=9, hidden_size=64, num_layers=1)

# Loss and optimizer
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [70]:
num_epochs = 30

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for batch_X, batch_y in train_loader: # mini- batch loop
        optimizer.zero_grad()
        preds = model(batch_X)
        loss = loss_fn(preds, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

Epoch 1/30, Loss: 0.5668
Epoch 2/30, Loss: 0.1564
Epoch 3/30, Loss: 0.1084
Epoch 4/30, Loss: 0.0836
Epoch 5/30, Loss: 0.0707
Epoch 6/30, Loss: 0.0626
Epoch 7/30, Loss: 0.0579
Epoch 8/30, Loss: 0.0553
Epoch 9/30, Loss: 0.0546
Epoch 10/30, Loss: 0.0542
Epoch 11/30, Loss: 0.0516
Epoch 12/30, Loss: 0.0500
Epoch 13/30, Loss: 0.0491
Epoch 14/30, Loss: 0.0484
Epoch 15/30, Loss: 0.0480
Epoch 16/30, Loss: 0.0477
Epoch 17/30, Loss: 0.0475
Epoch 18/30, Loss: 0.0472
Epoch 19/30, Loss: 0.0468
Epoch 20/30, Loss: 0.0464
Epoch 21/30, Loss: 0.0459
Epoch 22/30, Loss: 0.0452
Epoch 23/30, Loss: 0.0444
Epoch 24/30, Loss: 0.0435
Epoch 25/30, Loss: 0.0427
Epoch 26/30, Loss: 0.0416
Epoch 27/30, Loss: 0.0407
Epoch 28/30, Loss: 0.0397
Epoch 29/30, Loss: 0.0389
Epoch 30/30, Loss: 0.0380


In [71]:
model.eval()

preds, trues = [], []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        pred = model(batch_X)
        preds.append(pred.numpy())
        trues.append(batch_y.numpy())

preds = np.vstack(preds)
trues = np.vstack(trues)

mse = mean_squared_error(trues, preds)
print(f"Test MSE: {mse:.4f}")

Test MSE: 0.5688


In [72]:

r2 = r2_score(trues, preds)
print("R² score:", r2)

R² score: 0.4398761987686157


In [73]:
# MSE: .7995: 30 epochs, seq of 10, learning rate .001, adam optimizer, batch size 16 
# MSE: .5932: 30 epoches, seq of 25, bach size 16
# MSE: .6773: 30 epochs, seq of 25, batch size 32

Notes:
- Try graphing different batch sizes and how it affects accuracy
- cumulative line plot for high and low tide events
- increments of 10 up to 180